In real-world datasets, it is very common to encounter **missing values** — entries that are absent or not recorded during data collection. These gaps degrade the quality of machine learning models if not handled properly. This notebook documents the `MissingValueHandler` class from `ifri_mini_ml_lib` and compares each method with its scikit-learn equivalent.

## 1. Key Concepts

### 1.1 What is a Missing Value?

A **missing value** (`NaN` in Python) means that no valid data was recorded for a variable in a given observation.

### 1.2 Types of Missingness

| Type | Description | Titanic Example |
|------|-------------|------------------|
| **MCAR** | Missingness is unrelated to any data | A few `Embarked` values missing randomly |
| **MAR** | Missingness depends on other observed variables | `Age` missing more often in lower classes |
| **MNAR** | Missingness depends on the missing value itself | `Cabin` missing mostly for 3rd class passengers |

### 1.3 Key Terms

| Term | Definition |
|------|------------|
| **Imputation** | Replacing NaN with estimated values |
| **Deletion** | Removing rows/columns with too many NaN |
| **Statistical imputation** | Replacing with mean, median, or mode |
| **KNN imputation** | Estimation via the k nearest neighbors |
| **Regression imputation** | Prediction via a linear model |

## 2. Presentation of Algorithms

### 2.1 Deletion — `remove_missing(X, threshold, axis)`
Removes rows/columns exceeding the missing value threshold.
```
For each row in X:
    If proportion NaN > threshold → remove
Return cleaned dataset
```

### 2.2 Statistical Imputation — `impute_statistical(X, strategy)`
Replaces NaN with the mean, median, or mode of each column.
```
For each column in X:
    Compute statistic → replace NaN
Return imputed dataset
```

### 2.3 Default Value — `impute_default(X, value)`
Replaces all NaN with a fixed constant.
```
For each NaN cell → replace with constant
Return imputed dataset
```

### 2.4 KNN Imputation — `impute_knn(X, k, task)`
Estimates NaN from the k most similar rows.
```
For each column with NaN:
    Train KNN on complete rows
    Predict missing values
Return imputed dataset
```

### 2.5 Regression Imputation — `impute_regression(X, target_col)`
Predicts NaN of a target column via Linear Regression.
```
Train LinearRegression on complete rows
Predict missing values
Return imputed dataset
```

## 3. Implementation

We use the **Titanic dataset** which contains 177 missing values in `Age` and 2 in `Embarked` — an ideal real-world case study. For each method, the `ifri_mini_ml_lib` code is shown alongside its scikit-learn equivalent.

In [ ]:
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer, KNNImputer
from ifri_mini_ml_lib.preprocessing.preparation.missing_value_handler import MissingValueHandler

handler = MissingValueHandler()

titanic = sns.load_dataset('titanic')
df = titanic[['pclass', 'age', 'sibsp', 'parch', 'fare']].copy()
df.columns = ['Class', 'Age', 'Siblings', 'Parents', 'Fare']

print(df.head())
print(f"\nMissing values:\n{df.isnull().sum()}")

### Method 1 — Deletion

We remove Titanic rows where more than 20% of values are missing (threshold = 0.8). scikit-learn has no direct equivalent — we use pandas `dropna` as reference.

In [ ]:
# ifri_mini_ml_lib
df_ifri = handler.remove_missing(df, threshold=0.8, axis=0)
print(f"[IFRI]    {df.shape[0]} rows → {df_ifri.shape[0]} rows")

# sklearn (pandas dropna)
df_sk = df.dropna(thresh=int(0.8 * df.shape[1]))
print(f"[sklearn] {df.shape[0]} rows → {df_sk.shape[0]} rows")

### Method 2 — Statistical Imputation

We fill missing `Age` values with the mean and median. We compare `ifri_mini_ml_lib` results with sklearn's `SimpleImputer` and visualize the impact on the distribution.

In [ ]:
# ifri_mini_ml_lib
df_mean   = handler.impute_statistical(df.copy(), strategy='mean')
df_median = handler.impute_statistical(df.copy(), strategy='median')

# sklearn
sk_mean   = SimpleImputer(strategy='mean').fit_transform(df.values)
sk_median = SimpleImputer(strategy='median').fit_transform(df.values)

missing = df['Age'].isnull()
print(pd.DataFrame({
    'IFRI mean':    df_mean['Age'][missing].values[:5],
    'sklearn mean': sk_mean[missing, 0][:5]
}))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 6))

axes[0,0].hist(df['Age'].dropna(), bins=20, color='#95a5a6', alpha=0.6, label='Original')
axes[0,0].hist(df_mean['Age'],     bins=20, color='#3498db', alpha=0.7, label='IFRI')
axes[0,0].set_title('Mean — IFRI', fontweight='bold')
axes[0,0].legend()

axes[0,1].hist(df['Age'].dropna(), bins=20, color='#95a5a6', alpha=0.6, label='Original')
axes[0,1].hist(df_median['Age'],   bins=20, color='#3498db', alpha=0.7, label='IFRI')
axes[0,1].set_title('Median — IFRI', fontweight='bold')
axes[0,1].legend()

axes[1,0].hist(df['Age'].dropna(), bins=20, color='#95a5a6', alpha=0.6, label='Original')
axes[1,0].hist(sk_mean[:,0],       bins=20, color='#e67e22', alpha=0.7, label='sklearn')
axes[1,0].set_title('Mean — sklearn', fontweight='bold')
axes[1,0].legend()

axes[1,1].hist(df['Age'].dropna(), bins=20, color='#95a5a6', alpha=0.6, label='Original')
axes[1,1].hist(sk_median[:,0],     bins=20, color='#e67e22', alpha=0.7, label='sklearn')
axes[1,1].set_title('Median — sklearn', fontweight='bold')
axes[1,1].legend()

plt.suptitle('Age Distribution — IFRI (top) vs sklearn (bottom)', fontweight='bold')
plt.tight_layout()
plt.show()

### Method 3 — Default Value

We replace NaN with `-1` to explicitly signal their absence. We compare `impute_default` with sklearn's `SimpleImputer(strategy='constant')`.

In [ ]:
# ifri_mini_ml_lib
df_ifri_def = handler.impute_default(df.copy(), value=-1)

# sklearn
sk_def = SimpleImputer(strategy='constant', fill_value=-1).fit_transform(df.values)

print(pd.DataFrame({
    'IFRI':    df_ifri_def['Age'][missing].values[:5],
    'sklearn': sk_def[missing, 0][:5]
}))

### Method 4 — KNN Imputation

We estimate missing `Age` values using the 5 most similar passengers. We use the first 100 rows to limit computation time and compare `impute_knn` with sklearn's `KNNImputer`.

In [ ]:
subset = df.head(100).copy()
missing_knn = subset['Age'].isnull()

# ifri_mini_ml_lib
df_ifri_knn = handler.impute_knn(subset.copy(), k=5, task='regression')

# sklearn
sk_knn = KNNImputer(n_neighbors=5).fit_transform(subset.values)

print(pd.DataFrame({
    'IFRI KNN':    df_ifri_knn['Age'][missing_knn].values[:5],
    'sklearn KNN': sk_knn[missing_knn, 0][:5]
}))

### Method 5 — Regression Imputation

We predict missing `Age` values from `Class`, `Fare`, `Siblings`, `Parents`. We compare `impute_regression` with sklearn's `IterativeImputer` (still experimental).

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

missing_reg = df['Age'].isnull()

# ifri_mini_ml_lib
df_ifri_reg = handler.impute_regression(df.copy(), target_col='Age')

# sklearn
sk_reg = IterativeImputer(random_state=42).fit_transform(df.values)

print(pd.DataFrame({
    'IFRI regression':   df_ifri_reg['Age'][missing_reg].values[:5],
    'sklearn Iterative': sk_reg[missing_reg, 0][:5]
}))

## 4. Real-Life Applications

### 4.1 Healthcare and Medicine
In medical records, missing tests generate missing values. Poor imputation can distort a diagnosis or bias a disease prediction model.

### 4.2 Finance and Fraud Detection
Financial transactions may contain incomplete fields. Fraud detection systems must handle these missing values before analyzing suspicious patterns.

### 4.3 Recommender Systems
On Netflix or Amazon, the user-product matrix is very sparse. These NaN must be estimated to predict preferences and make relevant recommendations.

### 4.4 Weather Data and IoT Sensors
Sensor failures generate gaps in time series. Missing values must be interpolated to maintain continuity in climate analyses.

### 4.5 Surveys and Social Sciences
Respondents often skip sensitive questions (income, opinions). These non-responses are MNAR and their handling must be carefully considered to avoid bias.

## 5. Limitations and Challenges

### 5.1 Imputation Bias
Mean and median **artificially reduce data variance**, potentially underestimating real variability.

### 5.2 Missingness Mechanism Assumption
Most methods assume MCAR or MAR. If data is **MNAR**, imputation introduces systematic bias that is hard to correct.

### 5.3 Computational Cost
KNN and regression require training a model per column — **expensive** on large datasets with many incomplete columns.

### 5.4 Error Propagation
Imputing one column from others can **propagate errors** from one column to the next.

### 5.5 No Fit/Transform API
Unlike sklearn, `MissingValueHandler` recomputes statistics at every call, which can introduce **data leakage** when processing test data.

## 6. References

- Pedregosa et al. (2011). *Scikit-learn: Machine Learning in Python*. JMLR 12. https://scikit-learn.org
- Little, R.J.A. & Rubin, D.B. (2002). *Statistical Analysis with Missing Data* (2nd ed.). Wiley.
- Waskom, M. (2021). *Seaborn: statistical data visualization*. https://seaborn.pydata.org
- scikit-learn — SimpleImputer: https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html
- scikit-learn — KNNImputer: https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html
- scikit-learn — IterativeImputer: https://scikit-learn.org/stable/modules/generated/sklearn.impute.IterativeImputer.html
- Titanic Dataset: https://www.kaggle.com/c/titanic
- IFRI Mini ML Lib: https://github.com/IFRI-AI-Classes/ifri_mini_ml_lib